In [ ]:
!pip install transformers datasets librosa torch

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import librosa

import matplotlib.pyplot as plt

from transformers import AutoModelForAudioClassification, AutoFeatureExtractor
from datasets import load_dataset, Audio, DatasetDict

import torch.nn.functional as F

from scipy.stats import pearsonr, spearmanr
import pandas as pd
from scipy.interpolate import interp1d

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
TARGET_IDX = 35
save_dir = "/content/drive/MyDrive/Colab Notebooks/SER_Thesis/01_data2vec_TESS/attributions"

### Φόρτωση fine-tuned μοντέλου

In [ ]:
repo_id = "VasilisAsim/data2vec-finetuned-TESS_full"
model_ft = AutoModelForAudioClassification.from_pretrained(repo_id)
feature_extractor = AutoFeatureExtractor.from_pretrained(repo_id)
model_ft.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model_ft.to(device)

### Φόρτωση dataset και reference δείγματος

In [ ]:
def load_tess_speaker_independent():
    dataset = load_dataset("AbstractTTS/TESS", split="train")
    dataset = dataset.cast_column("audio", Audio(decode=False))

    def get_speaker(example):
        filename = example["audio"]["path"]
        if "OAF" in filename: return "OAF"
        elif "YAF" in filename: return "YAF"
        return "unknown"

    train_dataset = dataset.filter(lambda x: get_speaker(x) == "OAF")
    test_dataset = dataset.filter(lambda x: get_speaker(x) == "YAF")

    train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16_000))

    return DatasetDict({"train": train_dataset, "test": test_dataset})

dataset = load_tess_speaker_independent()
test_dataset = dataset["test"]

sample = test_dataset[TARGET_IDX]
audio = sample["audio"]["array"]
sampling_rate = sample["audio"]["sampling_rate"]
true_label = sample["emotion"]

# Αρχική πρόβλεψη (χωρίς διαταραχή): η κλάση-στόχος για όλα τα attributions
inputs = feature_extractor(audio, sampling_rate=sampling_rate, return_tensors="pt")
with torch.no_grad():
    predicted_class = model_ft(inputs["input_values"].to(device)).logits.argmax(-1).item()

print("True label:", true_label)
print("Predicted label:", model_ft.config.id2label[predicted_class])

### Φόρτωση αποθηκευμένων attributions

In [ ]:
original_attributions = {
    "Grad-CAM++": np.load(f"{save_dir}/Grad-CAM++_idx{TARGET_IDX}.npy"),
    "Saliency": np.load(f"{save_dir}/Saliency_idx{TARGET_IDX}.npy"),
}
for name, a in original_attributions.items():
    print(f"{name}: shape={a.shape}, sum={a.sum():.4f}")

### Συναρτήσεις attribution (επαναϋπολογισμός σε οποιοδήποτε waveform)

In [ ]:
class LogitsWrapper(nn.Module):
    """Wrapper που επιστρέφει τα raw logits του μοντέλου."""
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input_values):
        return self.model(input_values=input_values).logits


wrapped_model = LogitsWrapper(model_ft).to(device).eval()


def norm(x):
    """Κανονικοποίηση με το μέγιστο"""
    return x / (x.max() + 1e-8)


def resample_attribution(attr, target_frames=50):
    """
    Παίρνει την απόλυτη τιμή ενός attribution vector μεταβλητού μήκους και το
    επαναδειγματοληπτεί σε target_frames σημεία με γραμμική παρεμβολή,
    κανονικοποιώντας ώστε το άθροισμα να ισούται με 1.
    """
    attr = np.abs(attr)
    f = interp1d(np.linspace(0, 1, len(attr)), attr, kind="linear")
    resampled = f(np.linspace(0, 1, target_frames))
    return resampled / (resampled.sum() + 1e-10)


def compute_gradcampp(waveform, sampling_rate=16000, n_segments=50, target_class=None):
    """
    Grad-CAM++ στο τελευταίο transformer layer για οποιοδήποτε waveform.
    """
    inputs = feature_extractor(waveform, sampling_rate=sampling_rate, return_tensors="pt")
    input_values = inputs["input_values"].to(device)
    input_values.requires_grad_()

    target_layer = model_ft.data2vec_audio.encoder.layers[-1]
    acts, grads = {}, {}

    def fwd_hook(m, i, o):
        acts["A"] = o[0] if isinstance(o, tuple) else o

    def bwd_hook(m, gi, go):
        grads["dA"] = go[0]

    h1 = target_layer.register_forward_hook(fwd_hook)
    h2 = target_layer.register_full_backward_hook(bwd_hook)
    try:
        logits = wrapped_model(input_values)
        pred_class = logits.argmax(-1).item()
        class_idx = pred_class if target_class is None else target_class
        model_ft.zero_grad()
        logits[0, class_idx].backward()
    finally:
        # Αφαίρεση hooks, αλλιώς συσσωρεύονται σε κάθε κλήση
        h1.remove()
        h2.remove()

    A = acts["A"][0].detach()
    dA = grads["dA"][0].detach()

    g2, g3 = dA ** 2, dA ** 3
    denom = 2 * g2 + A.sum(0, keepdim=True) * g3
    denom = torch.where(denom != 0, denom, torch.ones_like(denom))
    alpha = g2 / denom
    w = (alpha * F.relu(dA)).sum(dim=0)
    cam = norm(F.relu((w * A).sum(dim=-1))).cpu().numpy()

    return resample_attribution(cam, n_segments), pred_class


def compute_saliency(waveform, sampling_rate=16000, n_segments=50, target_class=None):
    """
    Saliency map: το gradient του logit της κλάσης-στόχου ως προς το input.
    Επιστρέφει (attribution n_segments τιμών, προβλεπόμενη κλάση του waveform).
    """
    inputs = feature_extractor(waveform, sampling_rate=sampling_rate, return_tensors="pt")
    input_values = inputs["input_values"].to(device)
    input_values.requires_grad_()

    logits = wrapped_model(input_values)
    pred_class = logits.argmax(-1).item()
    class_idx = pred_class if target_class is None else target_class

    model_ft.zero_grad()
    logits[0, class_idx].backward()

    grad = input_values.grad[0].detach().cpu().numpy()
    return resample_attribution(grad, n_segments), pred_class


METHODS = {"Grad-CAM++": compute_gradcampp, "Saliency": compute_saliency}

### Έλεγχος αναπαραγωγής των αποθηκευμένων attributions

In [ ]:
for name, fn in METHODS.items():
    attr, pred = fn(audio, sampling_rate, target_class=predicted_class)
    r, _ = pearsonr(attr, original_attributions[name])
    print(f"{name}: Pearson r (νέος υπολογισμός vs αποθηκευμένο) = {r:.4f}")

In [ ]:
def jaccard_topk(a, b, k=10):
    """
    Δείκτης Jaccard μεταξύ των k segments με τη μεγαλύτερη σημασία σε δύο
    attribution διανύσματα: |A ∩ B| / |A ∪ B|.
    """
    top_a = set(np.argsort(a)[-k:])
    top_b = set(np.argsort(b)[-k:])
    return len(top_a & top_b) / len(top_a | top_b)


def attribution_metrics(attr, reference):
    """Pearson, Spearman και Jaccard top-10 ανάμεσα σε δύο διανύσματα attribution."""
    return {
        "pearson": pearsonr(attr, reference)[0],
        "spearman": spearmanr(attr, reference)[0],
        "jaccard": jaccard_topk(attr, reference, k=10),
    }


def add_noise_snr(waveform, snr_db, rng):
    """
    Προσθέτει Gaussian θόρυβο με δοσμένο SNR (σε dB) ως προς την ισχύ του
    σήματος.
    """
    noise_power = np.mean(waveform ** 2) / (10 ** (snr_db / 10))
    noise = rng.normal(0, np.sqrt(noise_power), len(waveform))
    return (waveform + noise).astype(np.float32)


def pitch_shift(waveform, n_steps, sr=16000):
    """Μετατόπιση του pitch κατά n_steps ημιτόνια (librosa)."""
    return librosa.effects.pitch_shift(waveform.astype(np.float32), sr=sr, n_steps=n_steps)

### Πείραμα 1: Gaussian θόρυβος (διαβαθμιζόμενο SNR)

In [ ]:
# Ισοδύναμο SNR των παλιών επιπέδων (απόλυτη τυπική απόκλιση), για συνέχεια με προηγούμενα αποτελέσματα
old_levels = [0.001, 0.003, 0.005, 0.01, 0.02]
print({lvl: round(10 * np.log10(np.mean(audio ** 2) / lvl ** 2), 1) for lvl in old_levels})

snr_levels = [40, 30, 20, 10, 5]   # dB· μικρότερο SNR = περισσότερος θόρυβος
n_draws = 10                        # ανεξάρτητες πραγματοποιήσεις θορύβου ανά επίπεδο

rows = []
for method, fn in METHODS.items():
    for snr in snr_levels:
        for draw in range(n_draws):
            perturbed = add_noise_snr(audio, snr, np.random.default_rng(draw))
            attr, pred = fn(perturbed, sampling_rate, target_class=predicted_class)
            rows.append({
                "method": method, "snr_db": snr, "draw": draw,
                **attribution_metrics(attr, original_attributions[method]),
                "class_changed": pred != predicted_class,
            })

df_noise = pd.DataFrame(rows)
summary_noise = df_noise.groupby(["method", "snr_db"]).agg(
    pearson_mean=("pearson", "mean"), pearson_std=("pearson", "std"),
    jaccard_mean=("jaccard", "mean"), jaccard_std=("jaccard", "std"),
    flip_rate=("class_changed", "mean"),
)
print(summary_noise.round(3))

### Πείραμα 2: Pitch shift

In [ ]:
pitch_steps = [-2, -1, -0.5, 0.5, 1, 2]   # ημιτόνια

rows = []
for method, fn in METHODS.items():
    for steps in pitch_steps:
        perturbed = pitch_shift(audio, steps, sr=sampling_rate)
        # Το pitch_shift μπορεί να αλλάξει ελαφρώς το μήκος: επαναφορά στο αρχικό
        if len(perturbed) != len(audio):
            perturbed = librosa.util.fix_length(perturbed, size=len(audio))

        attr, pred = fn(perturbed, sampling_rate, target_class=predicted_class)
        rows.append({
            "method": method, "pitch_steps": steps,
            **attribution_metrics(attr, original_attributions[method]),
            "predicted_class": model_ft.config.id2label[pred],
            "class_changed": pred != predicted_class,
        })

df_pitch = pd.DataFrame(rows)
print(df_pitch.round(3).to_string(index=False))

### Οπτικοποίηση αποτελεσμάτων

In [ ]:
colors = {"Grad-CAM++": "steelblue", "Saliency": "darkorange"}
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Θόρυβος: mean ± std ανά SNR ---
for method in METHODS:
    grouped = df_noise[df_noise["method"] == method].groupby("snr_db")
    for metric, linestyle in [("pearson", "-"), ("jaccard", "--")]:
        mean, std = grouped[metric].mean(), grouped[metric].std()
        axes[0].errorbar(mean.index, mean.values, yerr=std.values, marker="o", capsize=3,
                         linestyle=linestyle, color=colors[method], label=f"{method} – {metric}")
axes[0].invert_xaxis()
axes[0].set_xlabel("SNR (dB) — περισσότερος θόρυβος προς τα δεξιά")
axes[0].set_ylabel("Τιμή μετρικής")
axes[0].set_title("Σταθερότητα attribution υπό Gaussian θόρυβο")
axes[0].set_ylim(-1.05, 1.05)
axes[0].axhline(0, color="gray", linewidth=0.7, linestyle="--")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

# --- Pitch shift ---
for method in METHODS:
    d = df_pitch[df_pitch["method"] == method]
    axes[1].plot(d["pitch_steps"], d["pearson"], "-o", color=colors[method], label=f"{method} – pearson")
    axes[1].plot(d["pitch_steps"], d["jaccard"], "--s", color=colors[method], label=f"{method} – jaccard")
# Κόκκινη διακεκομμένη γραμμή όπου άλλαξε η πρόβλεψη του μοντέλου
for steps in df_pitch[df_pitch["class_changed"]]["pitch_steps"].unique():
    axes[1].axvline(x=steps, color="red", linestyle=":", alpha=0.5)
axes[1].set_xlabel("Μετατόπιση pitch (ημιτόνια)")
axes[1].set_ylabel("Τιμή μετρικής")
axes[1].set_title("Σταθερότητα attribution υπό pitch shift")
axes[1].set_ylim(-1.05, 1.05)
axes[1].axhline(0, color="gray", linewidth=0.7, linestyle="--")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
### Διαγνωστικό: Grad-CAM++ υπό pitch shift

for steps in [-2, -1, -0.5, 0.5, 1, 2]:
    perturbed = pitch_shift(audio, steps, sr=sampling_rate)
    if len(perturbed) != len(audio):
        perturbed = librosa.util.fix_length(perturbed, size=len(audio))

    # attr = ήδη κανονικοποιημένος χάρτης (n_segments τιμές)
    attr, pred = compute_gradcampp(perturbed, sampling_rate, target_class=predicted_class)
    print(f"{steps:+.1f} st | πρόβλεψη={model_ft.config.id2label[pred]} | "
          f"min={attr.min():.4g} max={attr.max():.4g} | μοναδικές={np.unique(attr).size}")